This is for Task-1

In [ ]:
import numpy as np
import pynq
from pynq import Overlay, allocate

# 1. Load the bitstream onto the FPGA
# Ensure 'my_design.bit' and 'my_design.hwh' are in this same directory!
overlay = Overlay("./my_design.bit")
dma = overlay.axi_dma_0

In [ ]:

# 2. Configuration Parameters
INPUT_HEX_FILE = "input_data_inv.hex"
OUTPUT_HEX_FILE = "processed_output_inv.hex"


In [ ]:
# 3. Read and Parse the Hex File into a NumPy Array
# This reads lines of hex strings (e.g., "0000000A") and converts them to 32-bit integers
print("Reading input hex file...")
with open(INPUT_HEX_FILE, "r") as f:
    hex_lines = [line.strip() for line in f if line.strip()]

num_elements = len(hex_lines)
print(f"Found {num_elements} data elements to process.")

In [ ]:
# Convert hex strings to unsigned 32-bit integers
input_data = np.array([int(val, 16) for val in hex_lines], dtype=np.uint32)


In [ ]:

# 4. Allocate Physically Contiguous DDR Memory Buffers
# Standard Python memory is fragmented; 'allocate' forces Linux to give us 
# a straight block of physical RAM addresses that the DMA hardware can burst-read.
in_buffer = allocate(shape=(num_elements,), dtype=np.uint32)
out_buffer = allocate(shape=(num_elements,), dtype=np.uint32)

# Copy our parsed hex data into the hardware-accessible input buffer
in_buffer[:] = input_data

In [ ]:

# 5. Execute the DMA Hardware Acceleration Loop
print("Launching hardware accelerator execution...")

# Open the receiver channel gate first so it's ready to catch incoming data
dma.recvchannel.transfer(out_buffer)

# Launch the transmitter channel to push the data from DDR to the PL fabric
dma.sendchannel.transfer(in_buffer)

# Block Python execution until the Receive DMA finishes writing the results back to DDR
dma.recvchannel.wait()
dma.sendchannel.wait()
print("Hardware processing completed successfully!")

In [ ]:
# 6. Write the Processed Data Back to a New Hex File
print(f"Writing results to {OUTPUT_HEX_FILE}...")
with open(OUTPUT_HEX_FILE, "w") as f:
    for value in out_buffer:
        # Formats the 32-bit unsigned integer back to an 8-character uppercase hex string
        f.write(f"{value:08X}\n")

print("Pipeline completely finished!")

In [ ]:
# 7. Clean up Memory Buffers (Good embedded practice to prevent memory leaks)
in_buffer.free()
out_buffer.free()